# 🏁 Module 3 — Class 6 LAB: End-to-End Olist Data Prep

**Lesson:** [bepro-aiml.github.io/aiml-platform/#/module/3/class/6](https://bepro-aiml.github.io/aiml-platform/#/module/3/class/6)

---

## 📖 The capstone

Today is the **biggest day of Module 3**. You combine everything from Classes 1-5 into one polished script that takes the 9 raw Olist CSVs and produces:

**`olist_clean.parquet`** ← the file every module from M4 to M8 will load

If you save this file correctly today, you have data ready for:
- M4 — supervised classifier predicting `is_late`
- M5 — customer RFM segmentation
- M6 — neural network on the same target
- M7 — sentiment-enriched model with review text
- M8 — deployed FastAPI endpoint

**Don't lose this file.** It's the spine of the next 6 weeks.

---

## ⏱ Time budget

**90 minutes.** Solo or in pairs.

## 🎯 Required output schema (21 columns)

| Column | Type | Source |
| --- | --- | --- |
| `order_id` | string | orders |
| `customer_unique_id` | string | customers |
| `customer_state` | string | customers |
| `seller_state` | string | sellers |
| `purchase_year` | int | engineered |
| `purchase_month` | int | engineered |
| `purchase_dayofweek` | int | engineered |
| `purchase_hour` | int | engineered |
| `is_weekend` | int (0/1) | engineered |
| `num_items` | int | items aggregate |
| `total_price` | float | items aggregate |
| `total_freight` | float | items aggregate |
| `log_freight` | float | engineered |
| `payment_type` | string | payments (primary) |
| `payment_installments` | int | payments (primary) |
| `distance_km` | float | Haversine on geolocation |
| `delivery_days` | float | engineered |
| `estimate_days` | float | engineered |
| `is_late` | int (0/1) | **TARGET** |
| `review_score` | int 1-5 | reviews |
| `review_comment_message` | string | reviews (kept for M7) |

**Filter to:** orders with `order_status` ∈ `['delivered', 'invoiced', 'shipped', 'approved']`. Keep dropped rows in a separate `orders_undelivered.parquet`.

---

## Stage 1 — Load (10 min)

In [ ]:
import pandas as pd, numpy as np

orders   = pd.read_csv('olist_data/olist_orders_dataset.csv')
customers = pd.read_csv('olist_data/olist_customers_dataset.csv')
items    = pd.read_csv('olist_data/olist_order_items_dataset.csv')
payments = pd.read_csv('olist_data/olist_order_payments_dataset.csv')
reviews  = pd.read_csv('olist_data/olist_order_reviews_dataset.csv')
products = pd.read_csv('olist_data/olist_products_dataset.csv')
sellers  = pd.read_csv('olist_data/olist_sellers_dataset.csv')
geo      = pd.read_csv('olist_data/olist_geolocation_dataset.csv')

for name, t in [('orders', orders), ('customers', customers), ('items', items),
                ('payments', payments), ('reviews', reviews),
                ('products', products), ('sellers', sellers)]:
    print(f'{name:10s}: {t.shape}')

## Stage 2 — Clean orders (15 min)

In [ ]:
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c], errors='coerce')

MODEL_STATUSES = {'delivered', 'invoiced', 'shipped', 'approved'}
orders_model = orders[orders['order_status'].isin(MODEL_STATUSES)].copy()
orders_undelivered = orders[~orders.index.isin(orders_model.index)].copy()
orders_undelivered.to_parquet('orders_undelivered.parquet', index=False)

before = len(orders_model)
orders_model = orders_model.dropna(subset=['order_delivered_customer_date',
                                          'order_estimated_delivery_date'])
print(f'Modellable rows: {len(orders_model):,} (dropped {before - len(orders_model):,})')

## Stage 3 — Aggregate items + payments (15 min)

In [ ]:
order_totals = items.groupby('order_id').agg(
    num_items=('order_item_id', 'count'),
    total_price=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
).reset_index()

primary_pay = (payments
    .sort_values('payment_value', ascending=False)
    .drop_duplicates('order_id')[['order_id', 'payment_type', 'payment_installments']])

first_seller = (items.sort_values(['order_id', 'order_item_id'])
                     .drop_duplicates('order_id')[['order_id', 'seller_id']])

## Stage 4 — Geolocation merge + Haversine (20 min)

In [ ]:
R = 6371
def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

geo_lookup = geo.groupby('geolocation_zip_code_prefix').agg(
    lat=('geolocation_lat', 'mean'),
    lon=('geolocation_lng', 'mean'),
).reset_index()

## Stage 5 — Master frame: merge everything + engineer all features (15 min)

In [ ]:
# Build the master frame from orders_model
df = orders_model.merge(customers[['customer_id', 'customer_unique_id',
                                   'customer_state', 'customer_zip_code_prefix']],
                        on='customer_id', how='left')
df = df.merge(order_totals, on='order_id', how='left')
df = df.merge(primary_pay, on='order_id', how='left')
df = df.merge(first_seller, on='order_id', how='left')
df = df.merge(sellers[['seller_id', 'seller_state', 'seller_zip_code_prefix']],
              on='seller_id', how='left')

# Customer lat/lon
df = df.merge(geo_lookup.rename(columns={'lat':'cust_lat', 'lon':'cust_lon'}),
              left_on='customer_zip_code_prefix',
              right_on='geolocation_zip_code_prefix', how='left')
df = df.drop(columns=['geolocation_zip_code_prefix'])

# Seller lat/lon
df = df.merge(geo_lookup.rename(columns={'lat':'sell_lat','lon':'sell_lon',
                                         'geolocation_zip_code_prefix':'sz'}),
              left_on='seller_zip_code_prefix', right_on='sz', how='left')
df = df.drop(columns=['sz'])

# Distance
df['distance_km'] = haversine(df['cust_lat'].values, df['cust_lon'].values,
                              df['sell_lat'].values, df['sell_lon'].values)

# Date features
df['delivery_days']      = (df['order_delivered_customer_date']
                            - df['order_purchase_timestamp']).dt.days
df['estimate_days']      = (df['order_estimated_delivery_date']
                            - df['order_purchase_timestamp']).dt.days
df['is_late']            = (df['order_delivered_customer_date']
                            > df['order_estimated_delivery_date']).astype(int)
df['purchase_year']      = df['order_purchase_timestamp'].dt.year
df['purchase_month']     = df['order_purchase_timestamp'].dt.month
df['purchase_dayofweek'] = df['order_purchase_timestamp'].dt.dayofweek
df['purchase_hour']      = df['order_purchase_timestamp'].dt.hour
df['is_weekend']         = (df['purchase_dayofweek'] >= 5).astype(int)
df['log_freight']        = np.log1p(df['total_freight'].fillna(0))

print(f'Master frame: {df.shape}')

## Stage 6 — Add reviews + save (10 min)

In [ ]:
reviews_first = (reviews.sort_values(['order_id', 'review_creation_date'])
                          .drop_duplicates('order_id')
                          [['order_id', 'review_score', 'review_comment_message']])
df = df.merge(reviews_first, on='order_id', how='left')

FINAL_COLS = [
    'order_id', 'customer_unique_id', 'customer_state', 'seller_state',
    'purchase_year', 'purchase_month', 'purchase_dayofweek', 'purchase_hour',
    'is_weekend',
    'num_items', 'total_price', 'total_freight', 'log_freight',
    'payment_type', 'payment_installments',
    'distance_km', 'delivery_days', 'estimate_days', 'is_late',
    'review_score', 'review_comment_message',
]
out = df[FINAL_COLS].copy()
out.to_parquet('olist_clean.parquet', index=False)

print(f'✅ olist_clean.parquet: {out.shape}')
print(f'   is_late rate:    {out["is_late"].mean():.4f} (expect ~0.07)')
print(f'   reviews present: {out["review_score"].notna().mean():.3f}')
print(f'\nColumns: {list(out.columns)}')

---

## Stage 7 — Findings (10 min)

**Write your findings here in this markdown cell:**

1. **Total rows in `olist_clean.parquet`:** ___
2. **`is_late` rate:** ___ (should be roughly 0.07)
3. **Three things you noticed during cleaning:**
   - ___
   - ___
   - ___
4. **One question for the M4 mentor:** ___

---

## 📤 Submission

Push **your notebook AND `olist_clean.parquet`** to:
```
module-3/class_6/submissions/<TeamName>/
```

## 🎓 Grading rubric (100 points)

| Component | Weight |
| --- | --- |
| Schema compliance (21 columns, correct dtypes) | 25 % |
| Cleaning decisions documented in markdown | 15 % |
| Pipeline / script is one-shot reproducible | 20 % |
| Haversine implementation correctness (within 1km of `geopy`) | 10 % |
| `is_late` rate in expected range (5-10%) | 10 % |
| Findings cell substantive | 10 % |
| Code quality + comments | 10 % |

## 🏆 Module 3 complete!

**You produced the spine of the entire compound project.** Every Olist module from M4 to M8 loads `olist_clean.parquet`. Your work today = the foundation of every team's resume bullet.

🏁 *See you in Module 4 — let's train our first model!*